# Prophet — Walmart Store Sales Forecasting

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/train.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/sampleSubmission.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/stores.csv
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/features.csv.zip
/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting/test.csv.zip


In [ ]:
try:
    from prophet import Prophet
except ImportError:
    %pip install -q prophet
    from prophet import Prophet

import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.CRITICAL)
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)

In [ ]:
!wget -q -O preprocessing.py "https://raw.githubusercontent.com/IrakliZerekidze/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting/main/preprocessing.py"
from preprocessing import run_pipeline, weighted_mae, SUPER_BOWL, LABOR_DAY, THANKSGIVING, CHRISTMAS

In [ ]:
%pip install -q dagshub mlflow

import dagshub
dagshub.init(repo_owner='izere23',
             repo_name='ML-Final-Walmart-Recruiting-Store-Sales-Forecasting',
             mlflow=True)

import mlflow
import mlflow.pyfunc
mlflow.set_experiment("Prophet_Training")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 36.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 37.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e186bc48-e817-437e-aa57-52b35c9348ea&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6136a1fd01d02505e8e0a0b821349d656aaa5ba099f958f2856af8a392697a33




Accessing as izere23

Initialized MLflow to track repo "izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting"

Repository izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/6760c8a9927b4ceda600ced0a2ba1c63', creation_time=1783821196107, effective_trace_archival_retention=None, experiment_id='7', last_update_time=1783821196107, lifecycle_stage='active', name='Prophet_Training', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

## მონაცემების ჩატვირთვა და train/validation split

In [5]:
train_part, valid_part, train_full, test_full = run_pipeline(
    data_dir="/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting",
    out_dir="/kaggle/working/processed",
)

for df in (train_part, valid_part, test_full):
    if "IsHoliday" not in df.columns and "IsHoliday_x" in df.columns:
        df.rename(columns={"IsHoliday_x": "IsHoliday"}, inplace=True)
        df.drop(columns=["IsHoliday_y"], inplace=True)

TARGET = "Weekly_Sales"
train = train_part.dropna(subset=[TARGET]).copy()
valid = valid_part.dropna(subset=[TARGET]).copy()

X_train, y_train = train.drop(columns=[TARGET]), train[TARGET]
X_val,   y_val   = valid.drop(columns=[TARGET]), valid[TARGET]

print("train:", X_train.shape, "| valid:", X_val.shape)

Expected rows if no gaps: 428409
Actual rows: 380107
Missing (Store,Dept,Date) combos filled in: 48302
Expected rows if no gaps: 476333
Actual rows: 421570
Missing (Store,Dept,Date) combos filled in: 54763
Saved parquet files to /kaggle/working/processed
train: (380107, 32) | valid: (41463, 31)


## Prophet-ის კონფიგურაცია და per-series forecaster

In [6]:
holidays = pd.concat([
    pd.DataFrame({"holiday": "superbowl",    "ds": pd.to_datetime(SUPER_BOWL)}),
    pd.DataFrame({"holiday": "labor_day",    "ds": pd.to_datetime(LABOR_DAY)}),
    pd.DataFrame({"holiday": "thanksgiving", "ds": pd.to_datetime(THANKSGIVING)}),
    pd.DataFrame({"holiday": "christmas",    "ds": pd.to_datetime(CHRISTMAS)}),
], ignore_index=True)

In [7]:
def fit_prophet_series(train_df, val_df, holidays, seasonality_mode="additive",
                       changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                       n_changepoints=25):
    """Fit one Prophet per (Store,Dept) on train, predict the val dates.
       Returns: preds aligned to val_df order, dict of fitted models, fallbacks."""
    from collections import defaultdict
    tr = train_df[["Store", "Dept", "Date"]].copy()
    tr["y"] = np.asarray(train_df["_y"])
    tr = tr.dropna(subset=["y"])
    tr_g = dict(tuple(tr.groupby(["Store", "Dept"])))
    g_mean = tr["y"].mean()

    val_idx = defaultdict(list)
    for i, (s, d) in enumerate(zip(val_df["Store"], val_df["Dept"])):
        val_idx[(s, d)].append(i)

    preds, models, fallback = np.full(len(val_df), np.nan), {}, {}
    done = 0
    for key, idxs in val_idx.items():
        hist  = tr_g.get(key)
        dates = val_df.iloc[idxs]["Date"]
        if hist is None or len(hist) < 10:
            fb = hist["y"].mean() if hist is not None and len(hist) else g_mean
            fallback[key] = fb; preds[idxs] = fb
            continue
        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                        daily_seasonality=False, holidays=holidays,
                        seasonality_mode=seasonality_mode,
                        changepoint_prior_scale=changepoint_prior_scale,
                        seasonality_prior_scale=seasonality_prior_scale,
                        n_changepoints=n_changepoints)
            m.fit(hist.rename(columns={"Date": "ds"})[["ds", "y"]])
            preds[idxs] = m.predict(pd.DataFrame({"ds": dates.values}))["yhat"].values
            models[key] = m
        except Exception:
            fb = hist["y"].mean(); fallback[key] = fb; preds[idxs] = fb
        done += 1
        if done % 250 == 0:
            print(f"  fit {done} series...")

    return np.clip(preds, 0, None), models, fallback, g_mean

In [ ]:
SEASONALITY_MODE = "multiplicative"
CPS, SPS, NCP = 0.05, 10.0, 25
RUN_NAME = "Prophet_multiplicative_capped"
USE_HOLIDAYS = holidays

va = valid[["Store", "Dept", "Date", "IsHoliday"]].copy().reset_index(drop=True)
va["_yt"] = y_val.values
tr_in = train.copy(); tr_in["_y"] = y_train.values

val_pred, models, fallback, g_mean = fit_prophet_series(
    tr_in, va, USE_HOLIDAYS, seasonality_mode=SEASONALITY_MODE,
    changepoint_prior_scale=CPS, seasonality_prior_scale=SPS, n_changepoints=NCP,
)

cap = (tr_in.groupby(["Store", "Dept"])["_y"].max() * 1.5).to_dict()
caps = np.array([cap.get((s, d), np.inf) for s, d in zip(va["Store"], va["Dept"])])
val_pred = np.minimum(val_pred, caps)

val_wmae = weighted_mae(va["_yt"].values, val_pred, va["IsHoliday"].values)
val_mae  = float(np.mean(np.abs(va["_yt"].values - val_pred)))
val_rmse = float(np.sqrt(np.mean((va["_yt"].values - val_pred) ** 2)))

from collections import defaultdict
tr_pred = np.full(len(tr_in), np.nan)
tr_idx = defaultdict(list)
for i, (s, d) in enumerate(zip(tr_in["Store"], tr_in["Dept"])):
    tr_idx[(s, d)].append(i)
for key, idxs in tr_idx.items():
    m = models.get(key); dates = tr_in.iloc[idxs]["Date"]
    tr_pred[idxs] = (m.predict(pd.DataFrame({"ds": dates.values}))["yhat"].values
                     if m is not None else fallback.get(key, g_mean))
tr_pred = np.clip(tr_pred, 0, None)
train_wmae = weighted_mae(tr_in["_y"].values, tr_pred, tr_in["IsHoliday"].values)

with mlflow.start_run(run_name=RUN_NAME):
    mlflow.log_param("model", "Prophet")
    mlflow.log_param("config", f"{SEASONALITY_MODE},capped_1.5x,holidays,per_store_dept")
    mlflow.log_param("seasonality_mode", SEASONALITY_MODE)
    mlflow.log_param("changepoint_prior_scale", CPS)
    mlflow.log_param("seasonality_prior_scale", SPS)
    mlflow.log_param("n_changepoints", NCP)
    mlflow.log_param("prediction_cap", "1.5x_series_hist_max")
    mlflow.log_param("holidays", "comp_holidays")
    mlflow.log_param("n_series_fit", len(models))
    mlflow.log_param("n_series_fallback", len(fallback))
    mlflow.log_metric("train_wmae", train_wmae)
    mlflow.log_metric("val_wmae", val_wmae)
    mlflow.log_metric("overfit_gap", val_wmae - train_wmae)
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_rmse", val_rmse)

print(f"[{RUN_NAME}]")
print(f"train WMAE : {train_wmae:.2f}")
print(f"val   WMAE : {val_wmae:.2f}")
print(f"overfit gap: {val_wmae - train_wmae:.2f}")
print(f"val MAE    : {val_mae:.2f}")
print(f"val RMSE   : {val_rmse:.2f}")
print(f"series fit : {len(models)}  |  fallback: {len(fallback)}")

  fit 250 series...
  fit 500 series...
  fit 750 series...
  fit 1000 series...
  fit 1250 series...
  fit 1500 series...
  fit 1750 series...
  fit 2000 series...
  fit 2250 series...
  fit 2500 series...
  fit 2750 series...
  fit 3000 series...
🏃 View run Prophet_multiplicative_capped at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/7/runs/d5e3562b1f654e0095c36f1948eb7094
🧪 View experiment at: https://dagshub.com/izere23/ML-Final-Walmart-Recruiting-Store-Sales-Forecasting.mlflow/#/experiments/7
[Prophet_multiplicative_capped]
train WMAE : 1314.95
val   WMAE : 1428.88
overfit gap: 113.93
val MAE    : 1449.56
val RMSE   : 2977.62
series fit : 3062  |  fallback: 42
